# 03 - Baseline Models

This notebook trains three simple baseline models for the binary heart disease classification task:

- Logistic Regression
- Random Forest
- XGBoost

The goal is to get a clean first modeling run and compare basic metrics. No hyperparameter tuning is performed here.

## 1. Imports and Paths

The models use the processed train/test files created by the preprocessing notebook.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

In [ ]:
PROCESSED_DATA_DIR = Path("../data/processed")

# Keep the notebook runnable if it is executed from the project root instead of notebooks/.
if not PROCESSED_DATA_DIR.exists():
    PROCESSED_DATA_DIR = Path("data/processed")

PROCESSED_DATA_DIR

WindowsPath('../data/processed')

## 2. Load Processed Data

The feature files already contain imputed numerical columns and one-hot encoded categorical columns. The label files contain the binary `target` column.

In [ ]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze("columns")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (736, 32)
X_test shape: (184, 32)
y_train shape: (736,)
y_test shape: (184,)


In [ ]:
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0
assert y_train.isna().sum() == 0
assert y_test.isna().sum() == 0

## 3. Define Baseline Models

These models use straightforward default-style settings. Random seeds are fixed for reproducibility, and Logistic Regression receives a higher `max_iter` only to avoid premature convergence issues.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
    ),
}

models

{'Logistic Regression': LogisticRegression(max_iter=5000),
 'Random Forest': RandomForestClassifier(n_jobs=-1, random_state=42),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='logloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=None, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=None, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=None, n_jobs=-1,
               num_parallel_tree=None, ...)}

## 4. Train and Evaluate

Each baseline is fit on the training data and evaluated on the held-out test set using accuracy, precision, recall, F1, and ROC-AUC.

In [ ]:
def evaluate_model(model_name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    }

In [ ]:
results = []

for model_name, model in models.items():
    results.append(evaluate_model(model_name, model, X_train, y_train, X_test, y_test))

results_df = pd.DataFrame(results)
results_df

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.842391,0.834862,0.892157,0.862559,0.925753
1,Random Forest,0.847826,0.849057,0.882353,0.865385,0.927726
2,XGBoost,0.858696,0.851852,0.901961,0.876190,0.901243


## 5. Baseline Comparison

The table below sorts the baseline results by ROC-AUC. This gives a quick reference point before moving on to more deliberate model development.

In [ ]:
results_df.sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Random Forest,0.847826,0.849057,0.882353,0.865385,0.927726
1,Logistic Regression,0.842391,0.834862,0.892157,0.862559,0.925753
2,XGBoost,0.858696,0.851852,0.901961,0.876190,0.901243
